In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
df= pd.read_csv("datasets.csv")

In [3]:
target = "price"

In [4]:
df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)
df['baths'] = df['baths'].replace('Not specified', None).astype(float)

In [5]:
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

df['last_review_year'] = df['last_review'].dt.year
df['last_review_month'] = df['last_review'].dt.month

C:\Users\rishi\AppData\Local\Temp\ipykernel_41936\1108888300.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')


In [6]:
# distance from center (approx)
center_lat, center_lon = df['latitude'].mean(), df['longitude'].mean()

df['distance_from_center'] = ((df['latitude'] - center_lat)**2 + 
                              (df['longitude'] - center_lon)**2)**0.5

In [7]:
df['host_total_listings'] = df['calculated_host_listings_count']

In [8]:
df['is_reviewed'] = df['number_of_reviews'] > 0

df['review_score'] = df['reviews_per_month'].fillna(0)

In [9]:
df =pd.get_dummies(df,columns=['room_type'], drop_first=True)

In [10]:
df.drop(['id','name','host_name','license'],axis=1,inplace=True)

In [17]:
for col in df.columns:
    try:
        df[col].astype(float)
    except:
        print('Problem column: ',col)

Problem column:  neighbourhood_group
Problem column:  neighbourhood
Problem column:  last_review
Problem column:  bedrooms


In [19]:
problem_cols = ['beds', 'bedrooms', 'rating', 'baths']  
for col in problem_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [20]:
df.fillna({
    'beds': df['beds'].median(),
    'bedrooms': df['bedrooms'].median(),
    'baths': df['baths'].median(),
    'rating': df['rating'].median()
}, inplace=True)

In [25]:
from sklearn.model_selection import train_test_split
X= df.drop('price', axis=1)
y=df['price']
X = pd.get_dummies(X, drop_first=True)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [27]:
X_train = X_train.drop(columns=['last_review'])
X_test = X_test.drop(columns=['last_review'])

In [29]:
print(X_train.isnull().sum().sort_values(ascending=False).head(10))

last_review_year                  5
availability_365                  5
host_total_listings               5
distance_from_center              5
last_review_month                 5
number_of_reviews_ltm             5
latitude                          5
calculated_host_listings_count    5
reviews_per_month                 5
number_of_reviews                 5
dtype: int64


In [30]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

X_train[num_cols] = X_train[num_cols].fillna(X_train[num_cols].median())
X_test[num_cols] = X_test[num_cols].fillna(X_train[num_cols].median())

In [32]:
# combine X and y temporarily
train_df = X_train.copy()
train_df['price'] = y_train

# drop rows where target is missing
train_df = train_df.dropna(subset=['price'])

# split back
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']

In [33]:
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [35]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [37]:
print("X_test NaN:", X_test.isnull().sum().sum())
print("y_test NaN:", y_test.isnull().sum())

X_test NaN: 0
y_test NaN: 6


In [38]:
test_df = X_test.copy()
test_df['price'] = y_test

test_df = test_df.dropna(subset=['price'])

X_test = test_df.drop('price', axis=1)
y_test = test_df['price']

In [39]:
from sklearn.metrics import mean_absolute_error, r2_score

def evaluate(model):
    preds = model.predict(X_test)
    print("MAE:", mean_absolute_error(y_test, preds))
    print("R2:", r2_score(y_test, preds))

print("Linear Regression")
evaluate(lr)

print("Random Forest")
evaluate(rf)

Linear Regression
MAE: 103.33174051103123
R2: 0.09203654783728943
Random Forest
MAE: 82.32960462873675
R2: -0.12894237845203138


In [41]:
import joblib

joblib.dump(rf, "airbnb_price_model.pkl")

['airbnb_price_model.pkl']

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor())
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', RandomForestRegressor())])